# Aprendizaje Automático I

## Trabajo Práctico: Predicción de lluvia en Australia

### Facultad de Ciencias Exactas, Ingeniería y Agrimensura (FCEIA) - Universidad Nacional de Rosario (UNR)

**Integrantes:** Tomás Gimbatti, Valentino Longo, Julián Paredes

---

### Contenidos

1. Configuración del entorno
2. Carga e inspección estructural del dataset
3. Gestión de datos nulos y duplicados
4. División train / test

## 1. Configuración del entorno

In [116]:
# Importamos las librerías necesarias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Semilla fija para que los resultados sean reproducibles
RANDOM_STATE = 42

# Contador para referenciar a los gráficos
N_FIG = 1

## 2. Carga e inspección estructural del dataset

**Diccionario de variables**

* **Date:** (Cualitativa) Fecha de la observación.
* **Location:** (Cualitativa) Nombre de la estación meteorológica. 
* **MinTemp:** (Continua) Temperatura mínima registrada en grados Celsius (°C) en las 24 horas previas a las 9:00 AM.
* **MaxTemp:** (Continua) Temperatura máxima registrada en grados Celsius (°C) en las 24 horas previas a las 9:00 AM.
* **Rainfall:** (Continua) Cantidad de lluvia recolectada (en milímetros) durante las 24 horas previas a las 9:00 AM.
* **Evaporation:** (Continua) Medida en milímetros (mm) acumulados en las 24 horas previas a las 9:00 AM.
* **Sunshine:** (Continua)  Número total de horas de sol brillante registradas durante todo el día.
* **WindGustDir:** (Cualitativa) Dirección de la ráfaga de viento más fuerte ocurrida en las 24 horas hasta la medianoche.
* **WindGustSpeed:** (Cualitativa) Velocidad de la ráfaga de viento más fuerte en kilómetros por hora (km/h) en las 24 horas anteriores a la medianoche.
* **WindDir9am:** (Cualitativa)  Dirección del viento registrada a las 9:00 AM.
* **WindDir3pm:** (Cualitativa)  Dirección del viento registrada a las 3:00 PM
* **WindSpeed9am:** (Cualitativa) Velocidad del viento en km/h medida como el promedio de los 10 minutos anteriores a las 9:00 AM.
* **WindSpeed3pm:** (Cualitativa) Velocidad del viento en km/h medida como el promedio de los 10 minutos anteriores a las 3:00 PM.
* **Humidity9am:** (Continua) Porcentaje de humedad relativa registrado a las 9:00 AM.
* **Humidity3pm:** (Continua) Porcentaje de humedad relativa registrado a las 3:00 PM.
* **Pressure9am:** (Continua) Presión atmosférica reducida al nivel del mar medida en hectopascales (hPa) a las 9:00 AM.
* **Pressure3pm:** (Continua) Presión atmosférica reducida al nivel del mar medida en hectopascales (hPa) a las 3:00 PM.
* **Cloud9am:** (Continua) Fracción de cielo cubierta por nubes a las 9:00 AM medido en octas (octavos de cielo cubierto).
* **Cloud3pm:** (Continua) Fracción de cielo cubierta por nubes a las 3:00 PM medido en octas (octavos de cielo cubierto).
* **Temp9am:** (Continua) Temperatura en grados Celsius (°C) medida a las 9:00 AM.
* **Temp3pm:** (Continua) Temperatura en grados Celsius (°C) medida a las 3:00 PM.
* **RainToday:** (Cualitativa Dicotómica) Toma el valor "Yes" si la precipitación (Rainfall) en las 24 horas previas a las 9:00 AM supera 1 mm, y "No" en caso contrario.
* **RainTomorrow:** (Cualitativa Dicotómica) Se calcula directamente a partir de la variable (RainfallTomorrow), si la lluvia del día de mañana supera 1.0 mm, toma el valor de "Yes" y "No" en caso contrario.
* **RainfallTomorrow:** (Cuantitativa) Cantidad de lluvia registrada al día siguiente en milímetros (mm).

In [117]:
# Lectura del archivo .csv
df = pd.read_csv("weatherAUS_2026C2.csv")

In [119]:
# Mostramos información del dataset sobre todas las columnas
print(f'Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas\n')
resumen = pd.DataFrame({
    "Cantidad de no nulos": df.notna().sum(),
    "Tipo de dato": df.dtypes
})

resumen.index.name = "Columna"
display(resumen)

Dimensiones del dataset: 145412 filas x 25 columnas



,Cantidad de no nulos,Tipo de dato
Columna,,
Unnamed: 0,145412,int64
Date,145412,str
Location,145412,str
MinTemp,143928,float64
MaxTemp,144159,float64
Rainfall,142152,float64
Evaporation,82658,float64
Sunshine,75616,float64
WindGustDir,135096,str


In [120]:
df.head()

,Unnamed: 0,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow,RainfallTomorrow
0,0,2008-12-01,Albury,13.6,22.5,0.6,NaN,NaN,W,45.0,...,23.8,1008.1,1007.2,8.0,NaN,16.8,20.8,No,No,0.0
1,1,2008-12-02,Albury,7.3,25.8,0.0,NaN,NaN,WNW,44.0,...,22.1,1011.1,1007.5,NaN,NaN,18.3,24.6,No,No,0.0
2,2,2008-12-03,Albury,13.2,26.2,0.0,NaN,NaN,WSW,46.0,...,31.5,1007.3,1009.4,NaN,2.0,21.2,23.4,No,No,0.0
3,3,2008-12-04,Albury,10.0,28.8,0.0,NaN,NaN,NE,23.0,...,18.5,1017.6,1012.4,NaN,NaN,17.4,26.3,No,No,1.0
4,4,2008-12-05,Albury,17.4,32.4,1.0,NaN,NaN,W,41.0,...,34.6,1010.9,1005.9,7.0,8.0,17.9,29.1,No,No,0.2


In [121]:
df.tail()

,Unnamed: 0,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow,RainfallTomorrow
145407,145454,2017-06-20,Uluru,2.6,22.0,0.0,NaN,NaN,E,29.0,...,26.5,1024.8,1021.2,NaN,NaN,10.1,21.6,No,No,0.0
145408,145455,2017-06-21,Uluru,3.1,23.2,0.0,NaN,NaN,E,31.0,...,24.7,1024.3,1020.5,NaN,NaN,11.1,21.5,No,No,0.0
145409,145456,2017-06-22,Uluru,3.6,26.0,0.0,NaN,NaN,NNW,21.0,...,21.4,1023.3,1018.4,NaN,NaN,11.8,24.5,No,No,0.0
145410,145457,2017-06-23,Uluru,4.8,27.4,0.0,NaN,NaN,N,39.0,...,23.7,1021.0,1016.0,NaN,NaN,12.0,26.9,No,No,0.0
145411,145458,2017-06-24,Uluru,7.5,26.5,0.0,NaN,NaN,SE,28.0,...,27.1,1019.8,1016.2,3.0,2.0,14.7,26.8,No,No,0.0


In [122]:
# Eliminamos la primera columna y la de RainFallTomorrow
df_limpio = df.drop(columns=["Unnamed: 0", "RainfallTomorrow"], errors="ignore").copy()


Decidimos eliminar "Unnamed: 0" ya que era una variable que no nos aportaba nada al actuar como índice. También eliminamos "RainfallTomorrow" porque generaría una fuga de datos al contener información sobre la precipitación del día de mañana que con datos nuevos esa variable no se conoce ya que no se puede medir la precipitación de un día que todavía no sucedió. 

In [123]:
# Mostramos 5 registros aleatorios con la columna eliminada
df_limpio.sample(5, random_state=RANDOM_STATE)

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
36428,2017-03-27,SydneyAirport,20.7,28.9,0.0,5.8,11.0,NE,52.0,NNW,...,73.9,67.2,1016.8,1011.9,1.0,1.0,23.8,27.2,No,No
131530,2013-06-28,Hobart,5.2,13.6,0.0,1.0,5.0,NW,30.0,NNW,...,75.0,59.8,1021.2,1021.9,NaN,NaN,7.4,12.1,No,No
30576,2009-03-16,Sydney,16.5,27.0,0.4,6.8,10.8,NaN,NaN,W,...,40.6,29.5,1011.1,1008.1,1.0,0.0,19.0,26.4,No,No
23078,2014-08-17,NorfolkIsland,10.1,17.7,0.6,3.0,6.1,E,36.0,E,...,59.4,57.2,1023.5,1020.5,7.0,7.0,14.6,15.8,No,No
123321,2016-03-12,Perth,18.4,33.2,0.0,8.4,11.2,SSW,40.0,ESE,...,56.0,47.6,1015.9,1012.2,0.0,0.0,25.4,28.7,No,No


In [124]:
# Separamos las variables por tipo
cols_numericas = df_limpio.select_dtypes(include=[np.number]).columns.tolist()
cols_categoricas = df_limpio.select_dtypes(include=['str']).columns.tolist()

print(f'Variables numéricas ({len(cols_numericas)}): {cols_numericas}')
print(f'Variables categóricas ({len(cols_categoricas)}): {cols_categoricas}')

Variables numéricas (16): ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
Variables categóricas (7): ['Date', 'Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday', 'RainTomorrow']


## 3. Gestión de datos nulos y duplicados

In [125]:
# Reporte de valores nulos por columna
nulos = df_limpio.isnull().sum()
pct_nulos = (nulos / len(df_limpio)) * 100

reporte_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct_nulos.round(2),
    'Tipo': df_limpio.dtypes
})

print('Reporte de Valores Nulos:')
print('=' * 50)
print(reporte_nulos)
print(f'\nTotal de celdas con nulos: {nulos.sum()} ({nulos.sum() / df_limpio.size * 100:.2f}% del dataset)')

Reporte de Valores Nulos:
               Nulos  Porcentaje (%)     Tipo
Date               0            0.00      str
Location           0            0.00      str
MinTemp         1484            1.02  float64
MaxTemp         1253            0.86  float64
Rainfall        3260            2.24  float64
Evaporation    62754           43.16  float64
Sunshine       69796           48.00  float64
WindGustDir    10316            7.09      str
WindGustSpeed  10253            7.05  float64
WindDir9am     10562            7.26      str
WindDir3pm      4226            2.91      str
WindSpeed9am    1767            1.22  float64
WindSpeed3pm    3061            2.11  float64
Humidity9am     2653            1.82  float64
Humidity3pm     4505            3.10  float64
Pressure9am    15061           10.36  float64
Pressure3pm    15024           10.33  float64
Cloud9am       55870           38.42  float64
Cloud3pm       59336           40.81  float64
Temp9am         1766            1.21  float64
Temp3pm 

In [126]:
# Observamos la cantidad de valores nulos por registro
nulos_por_registro = df_limpio.isna().sum(axis=1).value_counts().sort_index()

resumen_nulos = nulos_por_registro.rename_axis(
    "Cantidad de valores nulos"
).reset_index(name="Cantidad de registros")

display(resumen_nulos)

,Cantidad de valores nulos,Cantidad de registros
0,0,56412
1,1,10908
2,2,18336
3,3,9778
4,4,28666
5,5,4137
6,6,10220
7,7,2277
8,8,1071
9,9,491


In [127]:
# Porcentaje de registros que presentan algún valor nulo
nulos = resumen_nulos[resumen_nulos["Cantidad de valores nulos"] >= 1]["Cantidad de registros"].sum()
porcentaje_nulos = (nulos / len(df_limpio)) * 100
print(f"Porcentaje de registros que presentan algún valor nulo: {porcentaje_nulos.round(2)}%")

Porcentaje de registros que presentan algún valor nulo: 61.21%


In [128]:
# Analizamos porcentaje de datos duplicados
n_duplicados = df_limpio.duplicated().sum()
print(f'Registros duplicados: {n_duplicados} ({(n_duplicados / len(df_limpio) * 100).round(2)}%)')

Registros duplicados: 0 (0.0%)


In [ ]:
# Eliminamos los registros nulos de la variable target

registros_antes = len(df_limpio)
eliminados = df_limpio["RainTomorrow"].isna().sum()

df_limpio = df_limpio.dropna(subset=["RainTomorrow"]).copy()

X_sin_target = df_limpio.drop(columns=["RainTomorrow"])

valores_a_imputar = X_sin_target.isna().sum().sum()
registros_con_faltantes = X_sin_target.isna().any(axis=1).sum()

print(f"Registros eliminados por target nulo: {eliminados:,}")
print(f"Registros restantes: {len(df_limpio):,} de {registros_antes:,}")
print(f"Valores faltantes en variables predictoras por imputar: {valores_a_imputar:,}")
print(f"Registros restantes con al menos un faltante: {registros_con_faltantes:,}")

Registros eliminados por target nulo: 3,259
Registros restantes: 142,153 de 145,412
Valores faltantes en variables predictoras por imputar: 316,442
Registros restantes con al menos un faltante: 85,741


Elegimos eliminar los registros con valores nulso en el target porque sin variable objetivo, un registro no sirve para entrenar (no hay error que calcular) ni para evaluar (no hay valor real contra el cual comparar la predicción). Tampoco corresponde imputar el target: sería inventar la respuesta que el modelo debe aprender.

## 4. División train / test

Dividimos el dataset en entrenamiento (80%) y prueba (20%) antes de imputar y antes de seguir con el análisis. Todos los procedimientos que aprenden algo de los datos (la imputación, el escalado, los modelos) se ajustan únicamente con el conjunto de entrenamiento. Si el imputador se ajustara con todos los datos, la información del test se filtraría al modelo (data leakage) y la evaluación final dejaría de ser una medición honesta sobre datos nunca vistos.

Los registros con valores faltantes se reparten entre los dos conjuntos como cualquier otro registro, y se imputan después de la división. 

No divimos en train y test de manera aleatoria, ya que en ese caso podríamos predecir si el lunes llovío con la información entrenada del martes siguiente, generando fuga de datos. Lo que hicimos fue ordenar los datos por fecha y obtuvimos una fecha de corte para dividir al conjunto de datos en aproximadamente 80/20.

In [130]:
# Realizamos la división train-test buscando la fecha cuyo corte deja la proporción de filas más cercana a 80/20.
df_limpio["Date"] = pd.to_datetime(df_limpio["Date"], errors="raise")

filas_por_fecha = df_limpio.groupby("Date", sort=True).size()
filas_objetivo_train = 0.80 * len(df_limpio)
fecha_corte = (filas_por_fecha.cumsum() - filas_objetivo_train).abs().idxmin()

train = df_limpio.loc[df_limpio["Date"] <= fecha_corte].copy()
test = df_limpio.loc[df_limpio["Date"] > fecha_corte].copy()

print(f"Fecha de corte: {fecha_corte.date()}")
print(f"Train: {len(train):,} filas ({len(train) / len(df_limpio):.2%})")
print(f"Test:  {len(test):,} filas ({len(test) / len(df_limpio):.2%})")

Fecha de corte: 2015-11-08
Train: 113,701 filas (79.98%)
Test:  28,452 filas (20.02%)


In [131]:
X_train = train.drop(columns=["RainTomorrow"])
y_train = train["RainTomorrow"]

X_test = test.drop(columns=["RainTomorrow"])
y_test = test["RainTomorrow"]

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape} | y_test:  {y_test.shape}")

X_train: (113701, 22) | y_train: (113701,)
X_test:  (28452, 22) | y_test:  (28452,)
